# PARTIE 3 : SPARK SQL, DATAFRAMES ET OPTIMISATION CATALYST

## 1. Objectifs Pédagogiques
Dans cette partie, nous passons de l'API bas niveau (RDD) à l'API structurée (**Spark SQL et DataFrames**) :
1. **Schémas explicites (`StructType`)** : Éviter le coût du scan préliminaire d'inférence de schéma.
2. **Équivalence DataFrame API vs SQL déclaratif** : Utilisation de `groupBy().agg()` et de `createOrReplaceTempView()`.
3. **Jointures distribuées et optimisation Catalyst** : Mesure de performance entre jointure classique et `BroadcastHashJoin`.
4. **Benchmark UDF Python vs Expressions Natives (`when/otherwise`)** : Démontrer l'impact de la barrière JVM / Python.
5. **Stockage Lakehouse Silver** : Écriture au format Parquet optimisé et partitionné par ville (`city`).
6. **Analyse du plan physique avec `explain(True)`**.

## 2. Initialisation de la SparkSession connectée à MinIO

In [ ]:
import os
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col, sum as _sum, when, udf

# Configuration de l'environnement
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Partie3_SQL_DataFrames_Optimisation")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("[OK] SparkSession prête pour Spark SQL & Catalyst Optimizer.")

## 3. Schémas Explicites `StructType` (Sans Inférence)
**Pourquoi imposer un schéma explicite ?**  
L'inférence de schéma sur un gros volume JSON oblige Spark à lire l'intégralité du dataset une première fois pour deviner les types, doublant les I/O disque. Un schéma strict élimine ce scan et garantit l'intégrité des types.

In [ ]:
# Schéma officiel strict conforme au PDF
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount_mad", DoubleType(), True),
    StructField("items_count", IntegerType(), True),  # Corrigé : items_count avec underscore
    StructField("distance_km", DoubleType(), True),
    StructField("courier_id", StringType(), True),
    StructField("event_time", StringType(), True)
])

rest_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])

# Lecture directe depuis MinIO sans scan d'inférence
df_orders = spark.read.json("s3a://lake/bronze/history/orders_history.jsonl", schema=orders_schema)
df_rests = spark.read.json("s3a://lake/bronze/history/restaurants.json", schema=rest_schema, multiLine=True)

print(f"[OK] DataFrames chargés depuis MinIO : {df_orders.count():,} commandes, {df_rests.count()} restaurants.")

## 4. Chiffre d'Affaires par Ville : API DataFrame vs Spark SQL
Nous filtrons les commandes livrées (`DELIVERED`) et comparons la syntaxe de l'API DataFrame avec celle d'une vue SQL temporaire.

In [ ]:
df_delivered = df_orders.filter(col("status") == "DELIVERED")

# Approche 1 : API DataFrame
ca_df = (
    df_delivered
    .groupBy("city")
    .agg(_sum("amount_mad").alias("CA_Total"))
    .orderBy(col("CA_Total").desc())
)
print("--- CA par ville (API DataFrame) ---")
ca_df.show()

# Approche 2 : Spark SQL déclaratif via Temporary View
df_delivered.createOrReplaceTempView("orders_delivered_view")
ca_sql = spark.sql("""
    SELECT city, ROUND(SUM(amount_mad), 2) AS CA_Total
    FROM orders_delivered_view
    GROUP BY city
    ORDER BY CA_Total DESC
""")
print("--- CA par ville (Spark SQL TempView) ---")
ca_sql.show()

# Note pédagogique : Catalyst génère exactement le même plan physique pour les deux syntaxes !

## 5. Jointure Commandes x Restaurants & CA par Catégorie et Ville

In [ ]:
# Jointure Inner : df_orders.restaurant_id == df_rests.id
df_joined = (
    df_delivered
    .join(df_rests, df_delivered.restaurant_id == df_rests.id, "inner")
    .drop(df_rests.city)  # Supprime la colonne dupliquée 'city'
)

ca_category_city = (
    df_joined
    .groupBy("category", "city")
    .agg(_sum("amount_mad").alias("CA_MAD"))
    .orderBy(col("CA_MAD").desc())
)

print("--- Chiffre d'Affaires par Catégorie et par Ville (Top 10) ---")
ca_category_city.show(10)

## 6. Benchmark Réel : UDF Python vs Expression Native `when/otherwise`
Le professeur demande de catégoriser la distance en :
- `court` : < 2 km
- `moyen` : 2 à 5 km
- `long`  : > 5 km

Nous exécutons les deux versions et mesurons précisément leurs temps d'exécution.

In [ ]:
# 1. Implémentation via UDF Python classique
@udf(StringType())
def distance_band_udf(dist):
    if dist is None: return "inconnu"
    if dist < 2.0: return "court"
    elif dist <= 5.0: return "moyen"
    else: return "long"

# 2. Implémentation via fonctions natives Catalyst (when / otherwise)
df_native = df_joined.withColumn(
    "distance_band",
    when(col("distance_km") < 2.0, "court")
    .when(col("distance_km") <= 5.0, "moyen")
    .otherwise("long")
)

# --- Benchmark UDF ---
t0 = time.time()
df_with_udf = df_joined.withColumn("distance_band", distance_band_udf(col("distance_km")))
count_udf = df_with_udf.filter(col("distance_band") == "court").count()
duree_udf = time.time() - t0

# --- Benchmark Natif ---
t1 = time.time()
count_native = df_native.filter(col("distance_band") == "court").count()
duree_native = time.time() - t1

gain = (duree_udf / duree_native) if duree_native > 0 else 1.0
print(f"=== RÉSULTATS DU BENCHMARK UDF vs NATIF ===")
print(f" • Version UDF Python      : {duree_udf:.3f} secondes (count = {count_udf})")
print(f" • Version Native Catalyst : {duree_native:.3f} secondes (count = {count_native})")
print(f" • Facteur d'accélération  : {gain:.1f}x plus rapide avec l'expression native !")

### Pourquoi la version native est-elle tellement plus rapide ?
1. **La barrière JVM <-> Python** : Une UDF Python oblige Spark à sérialiser chaque ligne depuis la JVM vers un processus enfant Python via un pipe IPC (Inter-Process Communication), à exécuter le code Python, puis à désérialiser le résultat vers la JVM. Cette sérialisation/désérialisation détruit les performances.
2. **Optimisation Catalyst & Tungsten** : Avec `when/otherwise`, le code reste intégralement dans le moteur Spark (JVM). Catalyst optimise l'expression dans l'arbre d'exécution et Tungsten génère du bytecode Java natif à la volée (*Whole-Stage Code Generation*) opérant directement sur la mémoire off-heap binaire.

## 7. Écriture en Parquet dans MinIO (Couche Silver Partitionnée)
Nous sauvegardons les données nettoyées et enrichies au format Parquet dans `s3a://lake/silver/orders_history/`, partitionnées par ville (`city`).

In [ ]:
silver_path = "s3a://lake/silver/orders_history/"
print(f"Écriture des données enrichies dans {silver_path} (partitionné par city)...")

(
    df_native
    .write
    .mode("overwrite")
    .partitionBy("city")
    .parquet(silver_path)
)

print("[OK] Couche Silver Parquet écrite avec succès dans MinIO !")

## 8. Analyse du Plan d'Exécution Catalyst avec `explain(True)`

In [ ]:
print("=== ANALYSE EXPLAIN(TRUE) DE LA JOINTURE ===")
df_joined.explain(True)

### Interprétation des 4 Plans de Catalyst :
1. **Parsed Logical Plan** : Arbre syntaxique non résolu issu du code SQL / DataFrame (vérification de la grammaire).
2. **Analyzed Logical Plan** : Résolution des types de colonnes et des tables grâce au catalogue Spark.
3. **Optimized Logical Plan** : Optimisations logiques automatiques :
   - **Pushdown Predicates** : Le filtre `status = 'DELIVERED'` est poussé au plus tôt dès la lecture du fichier (`PushedFilters`).
   - **Column Pruning** : Seules les colonnes nécessaires sont lues.
4. **Physical Plan (`BroadcastHashJoin`)** : Le référentiel `restaurants` est très petit (200 lignes, quelques Ko). Catalyst choisit automatiquement la stratégie **BroadcastHashJoin** au lieu d'un coûteux SortMergeJoin. La petite table est diffusée en mémoire sur chaque nœud, évitant tout shuffle réseau pour la grande table de 50 000 commandes !